### Chapter 5.6 - Dropout

Dropout is a regularization method that randomly sets some hidden activations to zero during training. It makes the network less dependent on any single hidden unit.


In [ ]:
import math
import random
import numpy as np
import torch


### 5.6.1 Dropout in Practice

#### 1. Intuition

Dropout randomly removes activations during training. The dropout probability is the chance that an activation is set to zero.

During evaluation, dropout is turned off so predictions are deterministic.

#### 2. Why this exists

Dropout discourages hidden units from co-adapting too strongly. Co-adapting means relying on each other in a brittle way.


#### 3. Examples

Apply PyTorch dropout to a small tensor.


In [ ]:
torch.manual_seed(0)
drop = torch.nn.Dropout(p=0.5)
drop.train()
X = torch.ones(6)
drop(X)


Switch dropout to evaluation mode.


In [ ]:
drop.eval()
drop(X)


#### 4. Step-by-step breakdown

`p=0.5` means each activation has a 50 percent chance of being dropped during training.

`drop.train()` enables training behavior.

`drop.eval()` enables evaluation behavior.

In evaluation mode, the shown dropout layer leaves values unchanged.

#### 5. Connection to ML systems

PyTorch modules switch dropout behavior when the model uses `.train()` or `.eval()` mode.

#### 6. Common confusion points

- Dropout behaves differently during training and evaluation.
- Dropout is usually applied to hidden activations, not labels.
- Higher dropout means stronger noise.
- Dropout is a regularization method, not an optimizer.


### 5.6.2 Implementation from Scratch

#### 1. Intuition

From scratch, dropout creates a random mask of zeros and ones and multiplies activations by that mask.

A mask is a tensor used to keep or remove values.

#### 2. Why this exists

Manual dropout reveals that the method is just randomized elementwise filtering with scaling.


#### 3. Examples

Implement dropout for a small tensor.


In [ ]:
def dropout_layer(X, p):
    keep_prob = 1 - p
    mask = (torch.rand(X.shape) < keep_prob).float()
    return mask * X / keep_prob

X = torch.ones(5)
dropout_layer(X, 0.4)


#### 4. Step-by-step breakdown

`keep_prob` is the probability of keeping an activation.

`torch.rand(X.shape)` creates random values between 0 and 1.

The comparison creates a Boolean mask.

Dividing by `keep_prob` keeps the expected activation scale similar.

#### 5. Connection to ML systems

Framework dropout layers implement this idea and handle training/evaluation mode automatically.

#### 6. Common confusion points

- The mask is random during training.
- Scaling prevents average activation size from shrinking too much.
- `p=0` means no dropout.
- `p=1` would drop everything and is not useful here.


### 5.6.3 Concise Implementation

#### 1. Intuition

Concise dropout uses `torch.nn.Dropout` as a module inside a network.

It is usually placed after activation functions in hidden layers.

#### 2. Why this exists

The module version handles random masking and train/eval behavior consistently.


#### 3. Examples

Build a small MLP with dropout.


In [ ]:
net = torch.nn.Sequential(
    torch.nn.Linear(4, 5),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(5, 2),
)
net.train()
net(torch.randn(3, 4)).shape


#### 4. Step-by-step breakdown

The first linear layer creates hidden values.

ReLU applies nonlinearity.

Dropout randomly masks hidden activations during training mode.

The final linear layer produces outputs.

#### 5. Connection to ML systems

Dropout is common in fully connected networks, though modern architectures may rely on other regularization methods too.

#### 6. Common confusion points

- Dropout modules should be inside the model if they are part of training behavior.
- `.train()` and `.eval()` matter.
- Dropout changes outputs randomly during training.
- Do not use dropout as a substitute for validation.


### 5.6.4 Summary

#### 1. Intuition

Dropout randomly hides activations during training and is disabled during evaluation.

It regularizes by making the network less dependent on particular hidden units.

#### 2. Why this exists

Dropout is one practical tool for reducing overfitting in flexible neural networks.


#### 3. Examples

A dropout behavior summary.


In [ ]:
behavior = {
    "train_mode": "randomly mask activations",
    "eval_mode": "use all activations",
}
behavior


#### 4. Step-by-step breakdown

The dictionary contrasts training and evaluation behavior.

This mode difference is the most important operational detail.

Forgetting it can make evaluation noisy or training ineffective.

#### 5. Connection to ML systems

Model training scripts should call `.train()` for training and `.eval()` for validation or testing.

#### 6. Common confusion points

- Dropout is stochastic during training.
- Evaluation should be deterministic unless intentionally sampling.
- Dropout probability is a hyperparameter.
- Too much dropout can cause underfitting.


### 5.6.5 Exercises

#### 1. Intuition

These exercises practice dropout masks and modes.

#### 2. Why this exists

Understanding mode-dependent behavior prevents subtle evaluation bugs.


#### 3. Examples

Exercise 1: apply scratch dropout to a vector.


In [ ]:
torch.manual_seed(1)
X = torch.ones(4)
dropout_layer(X, 0.5)


Exercise 2: compare train and eval mode shape.


In [ ]:
net.eval()
out = net(torch.randn(2, 4))
out.shape


#### 4. Step-by-step breakdown

Exercise 1 checks random masking.

Exercise 2 checks that evaluation mode still preserves output shape.

The values may change by mode, but the shape should not.

#### 5. Connection to ML systems

These checks are useful when adding dropout to existing networks.

#### 6. Common confusion points

- Random masks differ unless the seed is fixed.
- Mode changes behavior, not tensor shape.
- Dropout is normally used only during training.
- Always set evaluation mode before validation metrics.
